# Clasificación de daños en puentes con CNN (VGG16) y transfer learning

**Contexto del caso:**
- No existía un dataset propio de imágenes de puentes: había que salir a buscar datos abiertos, y a proponer más de una opción antes de definir el enfoque.
- Se usó el enfoque de transfer learning sobre VGG16, entrenando solo el clasificador y dejando congelada la parte convolucional preentrenada en ImageNet.
- El primer intento, con el conjunto más amplio de clases disponible, dio resultados pobres en varias categorías. Eso no se escondió: se documentó como un hallazgo, y se ajustó el conjunto de clases para un segundo experimento más balanceado.
- Artículo completo, con el diagrama del pipeline: **https://fuzzyfrog.ai/es/ai-lab/proyectos/industria/clasificacion-danos-puentes-cnn-vgg16-deep-learning/**

**Nota sobre los datos:** este proyecto usa datos abiertos de inspección de puentes (dataset dacl1k y datasets del building-inspection-toolkit, ver referencias). Sustituye `data/` por tu propia carpeta de imágenes organizada por clase (formato `ImageFolder` de torchvision) si quieres reproducirlo con otro conjunto.


## Diagrama del pipeline

```
Imágenes de inspección (dacl1k + building-inspection-toolkit)
                    │
        Organizar en carpetas por clase (ImageFolder)
                    │
        VGG16 preentrenado en ImageNet
        (congelar capas convolucionales)
                    │
        Entrenar solo el clasificador (50 épocas, Adam)
                    │
        ┌───────────┴───────────┐
        │                       │
  Experimento 1             Experimento 2
  Conjunto amplio de        Conjunto refinado
  clases → resultados       de clases → resultados
  desbalanceados            balanceados
  (hallazgo documentado)    (matriz de confusión,
                             ROC, reporte de clasificación)
```

El diagrama interactivo con tooltip por bloque está embebido en el artículo de la plataforma.


## 1. Carga de datos

- Las imágenes se organizan en carpetas por clase (formato estándar `ImageFolder` de torchvision), un patrón que facilita reutilizar este mismo notebook con cualquier dataset de clasificación de imágenes.
- Se aplica resize a 256x256, normalización, y se puede tomar una fracción del dataset para iterar más rápido antes de correr con todo.


In [ ]:
import os
import torch
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms
import numpy as np

data_dir = "data/imagenes_puentes"  # una subcarpeta por clase

def cargar_dataset(data_dir, fraction=0.25, validation_split=0.2, clases_deseadas=None):
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ])

    dataset = datasets.ImageFolder(root=data_dir, transform=transform)

    if clases_deseadas is not None:
        indices_clase = [dataset.class_to_idx[c] for c in clases_deseadas]
        indices = [i for i, (_, label) in enumerate(dataset) if label in indices_clase]
        dataset = Subset(dataset, indices)

    if fraction < 1.0:
        n_subset = int(len(dataset) * fraction)
        dataset, _ = random_split(dataset, [n_subset, len(dataset) - n_subset])

    n_val = int(len(dataset) * validation_split)
    n_train = len(dataset) - n_val
    train_dataset, val_dataset = random_split(dataset, [n_train, n_val])

    return train_dataset, val_dataset


## 2. Explicación de los datos: por qué no se usaron todas las clases disponibles

El dataset abierto trae más categorías de daño de las que terminaron en el modelo final. Algunas clases tenían muy pocas imágenes bien etiquetadas, o se confundían visualmente con otras (por ejemplo, eflorescencia que tapa una grieta subyacente, algo que el propio paper de origen del dataset documenta como una fuente de error de etiquetado). Se decidió ser explícito sobre esto en vez de forzar el modelo a aprender clases sin suficiente señal.


## 3. Modelo: VGG16 con transfer learning

Condición del proyecto: no había capacidad de cómputo ni datos suficientes para entrenar una CNN desde cero. Se usó VGG16 preentrenado en ImageNet, congelando las capas convolucionales y entrenando solo el clasificador, adaptado al número de clases del proyecto.


In [ ]:
import torch.nn as nn
from torchvision.models import vgg16, VGG16_Weights

class ClasificadorDanosPuentes(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
        for parametro in self.features.parameters():
            parametro.requires_grad = False  # congelar la parte convolucional

        self.avgpool = nn.AdaptiveAvgPool2d((7, 7))
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


## 4. Experimento 1: conjunto amplio de clases

Se entrena primero con el conjunto de clases más amplio disponible en los datos abiertos, sin filtrar por calidad de etiquetado. El resultado, adelantado aquí, no fue bueno para varias clases, y eso es exactamente lo que se documenta a continuación en vez de ocultarlo.


In [ ]:
import torch.optim as optim
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def entrenar(modelo, train_loader, val_loader, num_epochs, criterion, optimizer):
    historial = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for epoch in range(num_epochs):
        modelo.train()
        running_loss, correct, total = 0.0, 0, 0
        for imagenes, etiquetas in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
            optimizer.zero_grad()
            salidas = modelo(imagenes)
            loss = criterion(salidas, etiquetas)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * imagenes.size(0)
            _, preds = torch.max(salidas, 1)
            correct += (preds == etiquetas).sum().item()
            total += etiquetas.size(0)

        historial["train_loss"].append(running_loss / total)
        historial["train_acc"].append(100 * correct / total)
    return historial

# Ejemplo de uso (ajusta clases_deseadas al esquema completo de tu dataset abierto):
# train_ds, val_ds = cargar_dataset(data_dir, clases_deseadas=None)  # None = todas las clases
# train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
# val_loader = DataLoader(val_ds, batch_size=32)
# modelo_1 = ClasificadorDanosPuentes(num_classes=len(train_ds.dataset.classes)).to(device)
# optimizer = optim.Adam(modelo_1.classifier.parameters(), lr=0.001)
# historial_1 = entrenar(modelo_1, train_loader, val_loader, num_epochs=50, criterion=nn.CrossEntropyLoss(), optimizer=optimizer)


## 5. Evaluación en dos capas: métricas y matriz de confusión

Igual que en otros proyectos de este laboratorio, una métrica global (accuracy) no basta. Aquí la matriz de confusión fue lo que reveló el problema real: la clase con más ejemplos limpios (grietas) llegó a 100% de aciertos, pero otras clases se confundían sistemáticamente entre sí.


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

def evaluar_modelo(modelo, val_loader, nombres_clases):
    modelo.eval()
    todas_etiquetas, todas_preds = [], []
    with torch.no_grad():
        for imagenes, etiquetas in val_loader:
            imagenes = imagenes.to(device)
            salidas = modelo(imagenes)
            _, preds = torch.max(salidas, 1)
            todas_preds.extend(preds.cpu().numpy())
            todas_etiquetas.extend(etiquetas.numpy())

    cm = confusion_matrix(todas_etiquetas, todas_preds)
    cm_pct = cm.astype("float") / cm.sum(axis=1, keepdims=True) * 100

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_pct, annot=True, fmt=".1f", cmap="Blues",
                xticklabels=nombres_clases, yticklabels=nombres_clases)
    plt.xlabel("Etiqueta predicha")
    plt.ylabel("Etiqueta verdadera")
    plt.title("Matriz de confusión (%)")
    plt.show()

    print(classification_report(todas_etiquetas, todas_preds, target_names=nombres_clases))
    return cm_pct


## 6. Experimento 2: conjunto refinado de clases

**Este es el ajuste central del proyecto.** En vez de forzar el modelo a aprender clases con poca señal, se refinó el conjunto a las cuatro categorías con suficiente cantidad y calidad de imágenes: Grieta, Eflorescencia, Descascaramiento y Barras Expuestas. El resultado mejoró de forma notoria y pareja entre clases, no solo en el promedio global.


In [ ]:
clases_refinadas = ["Grieta", "Eflorescencia", "Descascaramiento", "BarrasExpuestas"]

# train_ds, val_ds = cargar_dataset(data_dir, clases_deseadas=clases_refinadas)
# train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
# val_loader = DataLoader(val_ds, batch_size=32)
# modelo_2 = ClasificadorDanosPuentes(num_classes=len(clases_refinadas)).to(device)
# optimizer = optim.Adam(modelo_2.classifier.parameters(), lr=0.001)
# historial_2 = entrenar(modelo_2, train_loader, val_loader, num_epochs=50, criterion=nn.CrossEntropyLoss(), optimizer=optimizer)
# evaluar_modelo(modelo_2, val_loader, clases_refinadas)

# Resultado real documentado en la tesis (reporte de clasificación, conjunto refinado):
resultado_experimento_2 = {
    "Grieta":            {"precision": 0.80, "recall": 1.00, "f1": 0.89, "soporte": 42},
    "Eflorescencia":     {"precision": 0.82, "recall": 0.73, "f1": 0.77, "soporte": 49},
    "Descascaramiento":  {"precision": 0.70, "recall": 0.84, "f1": 0.76, "soporte": 31},
    "BarrasExpuestas":   {"precision": 0.62, "recall": 0.71, "f1": 0.67, "soporte": 28},
}
import pandas as pd
pd.DataFrame(resultado_experimento_2).T


## 7. Evaluación con imágenes nuevas

Además de las métricas sobre el set de validación, se probó el modelo con imágenes de puentes fuera del conjunto original, mostrando la imagen junto a las probabilidades predichas por clase. Esta validación cualitativa es la que un ingeniero civil realmente revisa antes de confiar en el modelo.


In [ ]:
def predecir_y_graficar(modelo, val_loader, nombres_clases, num_imagenes_por_clase=2):
    modelo.eval()
    mean, std = [0.5, 0.5, 0.5], [0.5, 0.5, 0.5]

    def desnormalizar(imagen):
        for c in range(3):
            imagen[c] = imagen[c] * std[c] + mean[c]
        return imagen

    with torch.no_grad():
        for imagenes, etiquetas in val_loader:
            imagenes = imagenes.to(device)
            salidas = modelo(imagenes)
            probs = torch.softmax(salidas, dim=1).cpu().numpy()
            for i in range(min(num_imagenes_por_clase, len(imagenes))):
                img = desnormalizar(imagenes[i].cpu().clone()).permute(1, 2, 0).numpy()
                plt.figure(figsize=(8, 3))
                plt.subplot(1, 2, 1)
                plt.imshow(img.clip(0, 1))
                plt.title(f"Clase verdadera: {nombres_clases[etiquetas[i]]}")
                plt.axis("off")
                plt.subplot(1, 2, 2)
                plt.barh(nombres_clases, probs[i])
                plt.xlim(0, 1)
                plt.title("Probabilidades predichas")
                plt.tight_layout()
                plt.show()
            break


## 8. Hallazgos principales

- No tener un dataset propio no es un obstáculo para hacer un proyecto real: se puede construir sobre datos abiertos bien documentados, como dacl1k y el building-inspection-toolkit.
- El primer experimento, con el conjunto amplio de clases, tuvo un desempeño pobre y desigual entre categorías. Eso no fue un fracaso: fue evidencia de que algunas clases no tenían suficiente señal limpia, algo que coincide con lo que el propio paper de origen del dataset reporta sobre desbalance y confusión entre clases.
- Refinar el conjunto a cuatro clases con buena cantidad y calidad de datos mejoró el desempeño de forma pareja, no solo en el promedio.
- Congelar la parte convolucional de VGG16 y entrenar solo el clasificador fue suficiente para este volumen de datos, sin necesidad de reentrenar toda la red.
